# Show, Attend and Tell - Colab Runbook

Use this notebook in Google Colab to install the repo, link Flickr8k from Drive, train, evaluate BLEU, and generate attention visualizations.

Before running: `Runtime -> Change runtime type -> GPU`.

## 1. Clone Repo

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Miwi343/Neural_Image_Caption_Generator.git"
REPO_DIR = Path("/content/Neural_Image_Caption_Generator")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=True)

## 2. Install Dependencies

In [ ]:
!python -m pip install -q -r requirements.txt

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 3. Mount Drive and Link Flickr8k

Put Flickr8k in Drive with this layout:

```text
flickr8k/
  images/
  captions.txt
  Flickr_8k.trainImages.txt
  Flickr_8k.devImages.txt
  Flickr_8k.testImages.txt
```

Edit `DRIVE_FLICKR8K_DIR` below if your folder is elsewhere.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import shutil
from pathlib import Path

DRIVE_FLICKR8K_DIR = Path("/content/drive/MyDrive/flickr8k")
DATA_ROOT = Path("data/flickr8k")

def replace_empty_placeholder_with_symlink(target: Path, source: Path):
    source = source.resolve()
    assert source.exists(), f"Dataset folder does not exist: {source}"

    if target.is_symlink():
        print(f"Already linked: {target} -> {target.resolve()}")
        return

    if target.exists():
        contents = list(target.iterdir())
        only_gitkeep = len(contents) == 1 and contents[0].name == ".gitkeep"
        if len(contents) == 0 or only_gitkeep:
            shutil.rmtree(target)
        else:
            raise RuntimeError(
                f"{target} already exists and is not empty. Move it aside or set DATA_ROOT manually."
            )

    target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, target)
    print(f"Linked: {target} -> {source}")

replace_empty_placeholder_with_symlink(DATA_ROOT, DRIVE_FLICKR8K_DIR)
!find data/flickr8k -maxdepth 2 | head -20

## 4. Persist Checkpoints and Results to Drive

This makes `checkpoints/` and `results/` write directly into Drive so Colab disconnects do not delete them.

In [ ]:
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/sat_results")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def link_output_dir(local_name: str):
    local = Path(local_name)
    remote = DRIVE_OUTPUT_DIR / local_name
    remote.mkdir(parents=True, exist_ok=True)

    if local.is_symlink():
        print(f"Already linked: {local} -> {local.resolve()}")
        return

    if local.exists():
        contents = list(local.iterdir())
        only_gitkeep = len(contents) == 1 and contents[0].name == ".gitkeep"
        if len(contents) == 0 or only_gitkeep:
            shutil.rmtree(local)
        else:
            print(f"Using existing local {local}; not replacing it.")
            return

    os.symlink(remote, local)
    print(f"Linked: {local} -> {remote}")

link_output_dir("checkpoints")
link_output_dir("results")

## 5. Validate Dataset and Run Tests

In [ ]:
from utils import validate_dataset_layout

validate_dataset_layout("data/flickr8k")
print("Flickr8k layout and split counts look good.")

In [ ]:
!pytest -q

## 6. Train

Default run uses `config.py` values. Pass flags to override key hyperparameters without editing any file:

| Flag | Default | What it controls |
|---|---|---|
| `--lambda_weight` | 1.0 | Doubly stochastic attention regularisation weight (Eq. 14) |
| `--finetune_epoch` | 5 | Epoch at which encoder unfreezes for fine-tuning |

In [ ]:
# Baseline run (config.py defaults)
!python train.py

# --- Uncomment one of these to train with tuned hyperparameters instead ---
# !python train.py --lambda_weight 0.5
# !python train.py --lambda_weight 0.5 --finetune_epoch 3
# !python train.py --finetune_epoch 10

Optional: inspect the validation log.

In [ ]:
import csv

log_path = Path("results/training_log.csv")
if log_path.exists():
    with open(log_path) as f:
        rows = list(csv.reader(f))
    for row in rows[-6:]:
        print(row)
else:
    print("No training log yet.")

## 7. Evaluate Test Set

Greedy decode (beam=1) baseline. See Section 10 below for a full beam-width + length-normalisation sweep.

In [ ]:
!python evaluate.py \
  --checkpoint checkpoints/best.pt \
  --data_root data/flickr8k \
  --vocab data/flickr8k/vocab.json \
  --split test \
  --beam_width 1 \
  --batch_size 64 \
  --results_out results/test_bleu_baseline.json

In [ ]:
import json

with open("results/test_bleu.json") as f:
    bleu = json.load(f)
bleu

## 8. Generate Attention Visualizations

In [ ]:
!python visualize.py \
  --checkpoint checkpoints/best.pt \
  --vocab data/flickr8k/vocab.json \
  --data_root data/flickr8k \
  --split test \
  --num_examples 6 \
  --output_dir results/attention_examples \
  --overlay_style paper \
  --dpi 200

In [ ]:
from IPython.display import Image, display

example_paths = sorted(Path("results/attention_examples").glob("*.png"))
print(f"Found {len(example_paths)} figures")
for path in example_paths[:6]:
    print(path)
    display(Image(filename=str(path)))

## 9. Final Output Locations

Because `checkpoints/` and `results/` were symlinked, the important files should already be in Drive under `sat_results/`:

```text
/content/drive/MyDrive/sat_results/checkpoints/best.pt
/content/drive/MyDrive/sat_results/results/training_log.csv
/content/drive/MyDrive/sat_results/results/test_bleu.json
/content/drive/MyDrive/sat_results/results/attention_examples/
```

Run the next cell as a backup copy for `vocab.json`.

In [ ]:
!mkdir -p "/content/drive/MyDrive/sat_results/data/flickr8k"
!cp data/flickr8k/vocab.json "/content/drive/MyDrive/sat_results/data/flickr8k/vocab.json"
!find "/content/drive/MyDrive/sat_results" -maxdepth 3 -type f | sort | head -50

## 10. Hyperparameter Experiments — Eval Sweep (no retraining)

Runs evaluate.py across every combination of beam width and length normalisation against the checkpoint already in `checkpoints/best.pt`. Prints a comparison table at the end.

In [ ]:
import json, subprocess
from pathlib import Path

eval_configs = [
    dict(beam_width=1, length_normalize=False),
    dict(beam_width=3, length_normalize=False),
    dict(beam_width=5, length_normalize=False),
    dict(beam_width=3, length_normalize=True),
    dict(beam_width=5, length_normalize=True),
]

sweep_results = []
for cfg in eval_configs:
    tag = f"beam{cfg['beam_width']}_lennorm{cfg['length_normalize']}"
    out = f"results/eval_{tag}.json"
    cmd = [
        "python", "evaluate.py",
        "--checkpoint", "checkpoints/best.pt",
        "--data_root", "data/flickr8k",
        "--vocab", "data/flickr8k/vocab.json",
        "--split", "test",
        "--batch_size", "64",
        "--beam_width", str(cfg["beam_width"]),
        "--results_out", out,
    ]
    if cfg["length_normalize"]:
        cmd.append("--length_normalize")
    subprocess.run(cmd, check=True)
    with open(out) as f:
        data = json.load(f)
    sweep_results.append({**cfg, **data["scores"]})

# Comparison table
print(f"\n{'beam':>5}  {'len_norm':>8}  {'BLEU-1':>7}  {'BLEU-2':>7}  {'BLEU-3':>7}  {'BLEU-4':>7}  {'METEOR':>7}")
print("-" * 65)
for r in sweep_results:
    meteor = f"{r['meteor']*100:>7.2f}" if "meteor" in r else "    N/A"
    print(f"{r['beam_width']:>5}  {str(r['length_normalize']):>8}  "
          f"{r['bleu1']*100:>7.2f}  {r['bleu2']*100:>7.2f}  "
          f"{r['bleu3']*100:>7.2f}  {r['bleu4']*100:>7.2f}  {meteor}")

## 11. Hyperparameter Experiments — Training Sweep (requires retraining)

Each cell below retrains with a different `lambda_weight` / `finetune_epoch` and evaluates with beam=5 + length normalisation. Results land in `results/` with a descriptive tag.

**Warning:** each run takes as long as the original training. Run one at a time, or pick the most promising variant based on the eval sweep above.

In [ ]:
import json, shutil, subprocess
from pathlib import Path

train_configs = [
    dict(label="lambda0.5",              lambda_weight=0.5, finetune_epoch=5),
    dict(label="lambda2.0",              lambda_weight=2.0, finetune_epoch=5),
    dict(label="finetune3",              lambda_weight=1.0, finetune_epoch=3),
    dict(label="finetune10",             lambda_weight=1.0, finetune_epoch=10),
    dict(label="lambda0.5_finetune3",    lambda_weight=0.5, finetune_epoch=3),
]

all_results = []
for cfg in train_configs:
    print(f"\n{'='*60}")
    print(f"Training: {cfg['label']}  (lambda={cfg['lambda_weight']}, finetune_epoch={cfg['finetune_epoch']})")
    print('='*60)

    subprocess.run([
        "python", "train.py",
        "--lambda_weight", str(cfg["lambda_weight"]),
        "--finetune_epoch", str(cfg["finetune_epoch"]),
    ], check=True)

    # Save this run's checkpoint under a unique name
    ckpt_tag = Path(f"checkpoints/best_{cfg['label']}.pt")
    shutil.copy("checkpoints/best.pt", ckpt_tag)

    out = f"results/eval_{cfg['label']}.json"
    subprocess.run([
        "python", "evaluate.py",
        "--checkpoint", str(ckpt_tag),
        "--data_root", "data/flickr8k",
        "--vocab", "data/flickr8k/vocab.json",
        "--split", "test",
        "--beam_width", "5",
        "--length_normalize",
        "--batch_size", "64",
        "--results_out", out,
    ], check=True)

    with open(out) as f:
        data = json.load(f)
    all_results.append({"label": cfg["label"], **data["scores"]})

# Summary table
print(f"\n{'Config':<25}  {'BLEU-1':>7}  {'BLEU-2':>7}  {'BLEU-3':>7}  {'BLEU-4':>7}  {'METEOR':>7}")
print("-" * 70)
for r in all_results:
    meteor = f"{r['meteor']*100:>7.2f}" if "meteor" in r else "    N/A"
    print(f"{r['label']:<25}  {r['bleu1']*100:>7.2f}  {r['bleu2']*100:>7.2f}  "
          f"{r['bleu3']*100:>7.2f}  {r['bleu4']*100:>7.2f}  {meteor}")